# SIPTA Notebook: Modelado e indicadores
Base para calcular indicadores, índices y reglas de priorización.

## Objetivos
- Calcular indicadores sectoriales por localidad.- Normalizar y construir el Índice de Prioridad Territorial (IPT).- Generar reglas simples de prioridad y evidencia.

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path('..').resolve()
PROCESSED_DIR = ROOT / 'data' / 'processed'

def load_master_table(filename: str = 'master_localidades.csv') -> pd.DataFrame:
    path = PROCESSED_DIR / filename
    assert path.exists(), f'No existe {path}'
    return pd.read_csv(path)

# master = load_master_table()


## Indicadores de ejemplo

In [ ]:
def camas_por_10000(df: pd.DataFrame, camas_col: str = 'camas', pop_col: str = 'poblacion') -> pd.Series:
    return (df[camas_col] / df[pop_col]) * 10000

def cupos_por_1000(df: pd.DataFrame, cupos_col: str = 'cupos', pop_obj_col: str = 'poblacion_objetivo') -> pd.Series:
    return (df[cupos_col] / df[pop_obj_col]) * 1000


## Normalización simple

In [ ]:
def min_max_normalize(series: pd.Series) -> pd.Series:
    if series.max() == series.min():
        return pd.Series(0.5, index=series.index)
    return (series - series.min()) / (series.max() - series.min())

def build_ipt(df: pd.DataFrame, components: dict) -> pd.Series:
    normalized = pd.DataFrame({name: min_max_normalize(df[col]) for name, col in components.items()})
    return normalized.mean(axis=1) * 100

# ipt = build_ipt(master, {'salud': 'salud_index', 'educacion': 'educacion_index'})


## Reglas de prioridad

In [ ]:
def simple_recommendation(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['recomendacion'] = 'Revisar prioridades'
    mask = df['ipt'] > 70
    df.loc[mask, 'recomendacion'] = 'Priorizar inversión y capacidad'
    return df


## Notas
- Actualizar los nombres de columnas al dataset real.- Documentar las dimensiones del IPT en `docs/manual_tecnico.md`.- Trasladar la lógica a `src/modeling/calculate_indicators.py`.